In [ ]:
import pandas as pd
from elasticsearch import Elasticsearch
import os
import numpy as np


from config import ELASTIC_PASSWORD,CERT_PATH

client = Elasticsearch(
    "https://localhost:9200",
    ca_certs=CERT_PATH,
    basic_auth=("elastic",ELASTIC_PASSWORD)
)

client.info()
index_name = "mondragon_trabajo_final"
csv_file = "df_meteorologico.csv"


# --- 2. LECTURA ---
try:
    df = pd.read_csv(csv_file, encoding='latin-1')
except:
    df = pd.read_csv(csv_file, encoding='cp1252')

if 'Unnamed: 0' in df.columns:
    df = df.drop(columns=['Unnamed: 0'])

# --- 3. LIMPIEZA DE DATOS (ETL) ---

# A) MAPA: Crear códigos ISO
mapa_provincias = {
    'GIPUZKOA': 'ES-SS', 'ARABA/ALAVA': 'ES-VI', 'BIZKAIA': 'ES-BI',
    'NAVARRA': 'ES-NA', 'MADRID': 'ES-M', 'VALENCIA': 'ES-V',
    'MALAGA': 'ES-MA', 'GRANADA': 'ES-GR', 'CORDOBA': 'ES-CO'
}
if 'region' in df.columns:
    df['province_iso'] = df['region'].astype(str).map(mapa_provincias)

# B) FECHAS PRINCIPALES
date_cols = ['booked_at', 'checkin_time', 'checkout_time', 'cancelled_at', 'F_C_I', 'F_C_O']
for col in date_cols:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors='coerce')

# C) DINERO (Quitamos comas)
num_cols = ['reservation_net_value', 'total_adr']
for col in num_cols:
    if col in df.columns:
        df[col] = df[col].astype(str).str.replace(',', '').apply(pd.to_numeric, errors='coerce')

# D) BOOLEANOS
bool_cols = ['all_entry_forms_completed', 'returning_inhabitant']
mapper = {'yes': True, 'no': False, True: True, False: False}
for col in bool_cols:
    if col in df.columns:
        df[col] = df[col].map(mapper)

# E) NULOS
df = df.replace({np.nan: None, pd.NaT: None})

# --- 4. MAPPING  ---
print("Creando estructura...")

properties = {
    "checkin_time": { "type": "date" },
    "booked_at": { "type": "date" },
    "province_iso": { "type": "keyword" },
    "city": { "type": "keyword" },
    "region": { "type": "keyword" },
    "business_segment": { "type": "keyword" },
    "total_adr": { "type": "float" },
    "reservation_net_value": { "type": "float" }
}

prefijos_clima = ['MIN_', 'MAX_', 'MEAN_', 'STD_', 'Q1_', 'Q2_', 'Q3_', 'IQR_'] # <--- F_C_ eliminado
for col in df.columns:
    if any(col.startswith(p) for p in prefijos_clima):
        properties[col] = { "type": "float" }

if client.indices.exists(index=index_name):
    client.indices.delete(index=index_name)
    print("Índice antiguo eliminado.")

client.indices.create(index=index_name, body={"mappings": {"properties": properties}})
print(" Índice nuevo creado.")

# --- 5. SUBIDA ---
print(f"Subiendo {len(df)} filas...")
contador = 0
errores = 0

for i, row in df.iterrows():
    try:
        doc = row.to_dict()
        for col in date_cols:
            if col in doc and doc[col] is not None:
                doc[col] = doc[col].isoformat()
        
        client.index(index=index_name, document=doc)
        contador += 1
        if contador % 500 == 0: print(f"{contador} filas subidas...")
            
    except Exception as e:
        errores += 1
        if errores < 5: 
            print(f"Error fila {i}: {e}")

print(f"\n FIN. Éxitos: {contador} | Errores: {errores}")
if errores == 0:
    print(" ¡Todo perfecto!.")